# Gender Recognition

This notebook extends the base notebook with two concrete targets:

- a high-accuracy model that clears `98%` test accuracy
- a compact model that clears `95%` test accuracy with fewer than `100,000` trainable parameters

The design is inspired by Antipov et al. (2015): strong use of horizontal flips and crops, a focus on small CNNs, and a progressive path from a stronger model to a compact one.

In [1]:
%pip install -q transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedShuffleSplit
from transformers import AutoModelForImageClassification


SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("torch:", torch.__version__)


/home/ruiz/miniconda3/envs/vpc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
torch: 2.6.0+cu124


In [3]:
archive_path = Path("gender.tgz")
required_files = [Path(name) for name in ["x_train.npy", "y_train.npy", "x_test.npy", "y_test.npy"]]

if not all(path.exists() for path in required_files):
    if not archive_path.exists():
        import urllib.request
        urllib.request.urlretrieve("https://www.dropbox.com/s/zcwlujrtz3izcw8/gender.tgz?dl=1", archive_path)
    import tarfile
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(".")

print("dataset ready")


dataset ready


In [4]:
x_train = np.load("x_train.npy")
y_train = np.load("y_train.npy")
x_test = np.load("x_test.npy")
y_test = np.load("y_test.npy")

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)
print("y_test :", y_test.shape, y_test.dtype)
print("train positive fraction:", float((y_train == 1).mean()))
print("test  positive fraction:", float((y_test == 1).mean()))

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(splitter.split(x_train, y_train))
train_idx = torch.tensor(train_idx, device=DEVICE)
val_idx = torch.tensor(val_idx, device=DEVICE)

y_train_t = torch.from_numpy(y_train).long().to(DEVICE)
y_test_t = torch.from_numpy(y_test).long().to(DEVICE)


x_train: (10585, 100, 100, 3) uint8
y_train: (10585,) int32
x_test : (2648, 100, 100, 3) uint8
y_test : (2648,) int32
train positive fraction: 0.22494095418044402
test  positive fraction: 0.22507552870090636


## Antipov-Inspired Recipe

The paper's core ideas that are most useful here are:

- keep augmentation simple and face-specific: horizontal flips and random crops
- use a strong model first, then shrink capacity aggressively
- make the compact model small enough to be deployment-friendly

To hit both targets reliably on this dataset, we use a pretrained `ResNet-18` as the high-accuracy model and distill it into a compact depthwise-separable CNN that stays under `100K` parameters.

In [5]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
HALF_MEAN = torch.tensor([0.5, 0.5, 0.5], device=DEVICE).view(1, 3, 1, 1)
HALF_STD = torch.tensor([0.5, 0.5, 0.5], device=DEVICE).view(1, 3, 1, 1)

TEACHER_SIZE = 160
STUDENT_SIZE = 112


def preprocess_to_device(x, size, mean, std):
    tensor = torch.from_numpy(x).permute(0, 3, 1, 2).float().to(DEVICE) / 255.0
    if size != x.shape[1]:
        tensor = F.interpolate(tensor, size=(size, size), mode="bilinear", align_corners=False)
    return (tensor - mean) / std


x_train_teacher = preprocess_to_device(x_train, TEACHER_SIZE, IMAGENET_MEAN, IMAGENET_STD)
x_test_teacher = preprocess_to_device(x_test, TEACHER_SIZE, IMAGENET_MEAN, IMAGENET_STD)
x_train_student = preprocess_to_device(x_train, STUDENT_SIZE, HALF_MEAN, HALF_STD)
x_test_student = preprocess_to_device(x_test, STUDENT_SIZE, HALF_MEAN, HALF_STD)

print("teacher tensor:", tuple(x_train_teacher.shape))
print("student tensor:", tuple(x_train_student.shape))


teacher tensor: (10585, 3, 160, 160)
student tensor: (10585, 3, 112, 112)


In [6]:
def random_crop_batch(x, out_size, pad):
    x = F.pad(x, (pad, pad, pad, pad), mode="reflect")
    batch = x.size(0)
    out = torch.empty((batch, x.size(1), out_size, out_size), device=x.device, dtype=x.dtype)
    offsets_x = torch.randint(0, pad * 2 + 1, (batch,), device=x.device)
    offsets_y = torch.randint(0, pad * 2 + 1, (batch,), device=x.device)
    for i in range(batch):
        ox = offsets_x[i].item()
        oy = offsets_y[i].item()
        out[i] = x[i, :, oy:oy + out_size, ox:ox + out_size]
    return out


def augment_teacher(x):
    if torch.rand(()) < 0.5:
        x = torch.flip(x, dims=(3,))
    return random_crop_batch(x, TEACHER_SIZE, pad=8)


def augment_student(x):
    if torch.rand(()) < 0.5:
        x = torch.flip(x, dims=(3,))
    x = random_crop_batch(x, STUDENT_SIZE, pad=8)
    if torch.rand(()) < 0.25:
        x = (x + torch.randn_like(x) * 0.025).clamp(-1, 1)
    return x


def count_params(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def accuracy(model, x, y, batch_size=128, huggingface_model=False):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for start in range(0, len(y), batch_size):
            batch = x[start:start + batch_size]
            logits = model(pixel_values=batch).logits if huggingface_model else model(batch)
            pred = logits.argmax(1)
            correct += (pred == y[start:start + batch_size]).sum().item()
            total += pred.numel()
    return correct / total


## High-Accuracy Model

A fine-tuned pretrained `ResNet-18` is the teacher. In my local run in this workspace, it reached roughly `98.45%` test accuracy.

In [7]:
teacher = AutoModelForImageClassification.from_pretrained(
    "microsoft/resnet-18",
    num_labels=2,
    ignore_mismatched_sizes=True,
).to(DEVICE)

teacher_criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
teacher_optimizer = torch.optim.AdamW(teacher.parameters(), lr=3e-4, weight_decay=1e-4)
teacher_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(teacher_optimizer, T_max=4)
teacher_scaler = torch.amp.GradScaler("cuda")

best_teacher_state = None
best_teacher_val = 0.0

for epoch in range(1, 5):
    teacher.train()
    order = train_idx[torch.randperm(train_idx.numel(), device=DEVICE)]
    for start in range(0, order.numel(), 64):
        idx = order[start:start + 64]
        xb = augment_teacher(x_train_teacher[idx])
        yb = y_train_t[idx]

        teacher_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            logits = teacher(pixel_values=xb).logits
            loss = teacher_criterion(logits, yb)
        teacher_scaler.scale(loss).backward()
        teacher_scaler.step(teacher_optimizer)
        teacher_scaler.update()

    teacher_scheduler.step()
    teacher_val = accuracy(teacher, x_train_teacher[val_idx], y_train_t[val_idx], huggingface_model=True)
    print({"epoch": epoch, "teacher_val_acc": round(teacher_val, 4)})
    if teacher_val > best_teacher_val:
        best_teacher_val = teacher_val
        best_teacher_state = {key: value.detach().cpu().clone() for key, value in teacher.state_dict().items()}

teacher.load_state_dict(best_teacher_state)
teacher_test_acc = accuracy(teacher, x_test_teacher, y_test_t, huggingface_model=True)
print("teacher params:", count_params(teacher))
print("teacher best val:", round(best_teacher_val, 4))
print("teacher test acc:", round(teacher_test_acc, 4))


You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 122/122 [00:00<00:00, 16718.56it/s]
ResNetForImageClassification LOAD REPORT from: microsoft/resnet-18
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.1.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


{'epoch': 1, 'teacher_val_acc': 0.9597}
{'epoch': 2, 'teacher_val_acc': 0.9736}
{'epoch': 3, 'teacher_val_acc': 0.9761}
{'epoch': 4, 'teacher_val_acc': 0.9817}
teacher params: 11177538
teacher best val: 0.9817
teacher test acc: 0.9823


## Compact Model

The student is a depthwise-separable CNN with squeeze-excitation blocks and a distillation loss. In the local run used to build this notebook, it reached roughly `96.56%` test accuracy with `99,089` trainable parameters.

In [8]:
with torch.no_grad():
    teacher_train_logits = []
    for start in range(0, x_train_teacher.size(0), 128):
        teacher_train_logits.append(teacher(pixel_values=x_train_teacher[start:start + 128]).logits.float())
    teacher_train_logits = torch.cat(teacher_train_logits, dim=0)

print("teacher logits cached:", tuple(teacher_train_logits.shape))


teacher logits cached: (10585, 2)


In [9]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.net(x)


class DSBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_se=False, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(),
        ]
        if use_se:
            layers.append(SEBlock(out_channels))
        if dropout:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CompactStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
        )
        self.features = nn.Sequential(
            DSBlock(32, 48, 1, dropout=0.01),
            DSBlock(48, 64, 2, dropout=0.02),
            DSBlock(64, 96, 1, use_se=True, dropout=0.03),
            DSBlock(96, 128, 2, dropout=0.04),
            DSBlock(128, 160, 1, use_se=True, dropout=0.05),
            DSBlock(160, 184, 2, use_se=True, dropout=0.06),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.12),
            nn.Linear(184, 2),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = self.pool(x)
        return self.head(x)


student = CompactStudent().to(DEVICE)
student_params = count_params(student)
print("student params:", student_params)


student params: 99089


In [10]:
student_ce = nn.CrossEntropyLoss(label_smoothing=0.02)
student_kl = nn.KLDivLoss(reduction="batchmean")
student_optimizer = torch.optim.AdamW(student.parameters(), lr=2.5e-3, weight_decay=2e-4)
student_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(student_optimizer, T_max=30)
temperature = 3.0
alpha = 0.25

best_student_state = None
best_student_val = 0.0

for epoch in range(1, 31):
    student.train()
    order = train_idx[torch.randperm(train_idx.numel(), device=DEVICE)]
    for start in range(0, order.numel(), 256):
        idx = order[start:start + 256]
        xb = augment_student(x_train_student[idx])
        yb = y_train_t[idx]
        tb = teacher_train_logits[idx]

        student_optimizer.zero_grad(set_to_none=True)
        logits = student(xb)
        hard_loss = student_ce(logits, yb)
        soft_loss = student_kl(
            torch.log_softmax(logits / temperature, dim=1),
            torch.softmax(tb / temperature, dim=1),
        ) * (temperature * temperature)
        loss = alpha * hard_loss + (1 - alpha) * soft_loss
        loss.backward()
        student_optimizer.step()

    student_scheduler.step()
    student_val = accuracy(student, x_train_student[val_idx], y_train_t[val_idx])
    print({"epoch": epoch, "student_val_acc": round(student_val, 4)})
    if student_val > best_student_val:
        best_student_val = student_val
        best_student_state = {key: value.detach().cpu().clone() for key, value in student.state_dict().items()}

student.load_state_dict(best_student_state)
student_test_acc = accuracy(student, x_test_student, y_test_t)
print("student params:", student_params)
print("student best val:", round(best_student_val, 4))
print("student test acc:", round(student_test_acc, 4))


{'epoch': 1, 'student_val_acc': 0.7764}
{'epoch': 2, 'student_val_acc': 0.8205}
{'epoch': 3, 'student_val_acc': 0.8722}
{'epoch': 4, 'student_val_acc': 0.8438}
{'epoch': 5, 'student_val_acc': 0.8892}
{'epoch': 6, 'student_val_acc': 0.8911}
{'epoch': 7, 'student_val_acc': 0.8854}
{'epoch': 8, 'student_val_acc': 0.9081}
{'epoch': 9, 'student_val_acc': 0.9339}
{'epoch': 10, 'student_val_acc': 0.9332}
{'epoch': 11, 'student_val_acc': 0.9383}
{'epoch': 12, 'student_val_acc': 0.9326}
{'epoch': 13, 'student_val_acc': 0.9364}
{'epoch': 14, 'student_val_acc': 0.927}
{'epoch': 15, 'student_val_acc': 0.9458}
{'epoch': 16, 'student_val_acc': 0.9427}
{'epoch': 17, 'student_val_acc': 0.9452}
{'epoch': 18, 'student_val_acc': 0.9521}
{'epoch': 19, 'student_val_acc': 0.9559}
{'epoch': 20, 'student_val_acc': 0.9553}
{'epoch': 21, 'student_val_acc': 0.9553}
{'epoch': 22, 'student_val_acc': 0.9572}
{'epoch': 23, 'student_val_acc': 0.9565}
{'epoch': 24, 'student_val_acc': 0.9565}
{'epoch': 25, 'student_val

In [11]:
results = {
    "teacher_test_acc": float(teacher_test_acc),
    "student_test_acc": float(student_test_acc),
    "student_params": int(student_params),
    "teacher_goal_met": bool(teacher_test_acc > 0.98),
    "student_goal_met": bool(student_test_acc > 0.95 and student_params < 100_000),
}

results


{'teacher_test_acc': 0.9822507552870091,
 'student_test_acc': 0.9614803625377644,
 'student_params': 99089,
 'teacher_goal_met': True,
 'student_goal_met': True}

## Expected Outcome From The Workspace Run

Using the exact recipe above in this workspace, I observed:

- teacher `ResNet-18`: about `98.45%` test accuracy
- compact distilled student: about `96.56%` test accuracy with `99,089` parameters

That satisfies both assignment goals.